# Offline load balancing

In @ref tutorials_exahype2_30_domain-decomposition, the load balancing decides
at runtime when and how to split up the mesh. Such generic strategies work for
all kinds of meshes, but they know little about the particular experiment, and
they have to find out about the mesh while it is constructed.

If we know the mesh before we run the code, we can compute the decomposition
ahead of time, i.e. offline, and hard-code the outcome. This is particularly
simple for static meshes. Offline load balancing is a straightforward way to
get a reasonable decomposition for a particular experiment, and it can serve as
a baseline for something more bespoke and dynamic later on. In this tutorial, we

- compute the decomposition of the regular mesh of @ref tutorials_exahype2_30_domain-decomposition with Peano's offline load balancing,
- hard-code the outcome through the strategy toolbox::loadbalancing::strategies::Hardcoded, and
- visualise the resulting subdomains with ParaView.

@ref page_toolbox_loadbalancing_api "The load balancing API" documents all
the Python routines that we use below.

## The solver

We use exactly the same Euler solver as in @ref tutorials_exahype2_30_domain-decomposition:

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../src"))

import peano4.api
import exahype2

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exahype2", "offlineloadbalancing"],
    project_name = "Euler",
    directory    = "35_offline-load-balancing",
    executable   = "euler",
)

patch_size = 9

solver = exahype2.api.Solver(
    "EulerFV",
    4,   # unknowns
    0,   # auxiliary variables
)
solver.add_pde_term(exahype2.api.Flux(stateless=False))
solver.add_pde_term(exahype2.api.MaxEigenvalue(stateless=False))
solver.add_pde_term(exahype2.api.InitPointWisely(stateless=False))

solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(solver, patch_size)
solver.grid_construction = exahype2.api.fv.SetupSimulationOverRegularGrid(solver, volume_size=0.001)
solver.plotter           = exahype2.api.fv.PlotAsVTKFile(
    solver,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.01,
)

project.add_solver(solver)

project.set_global_simulation_parameters(
    dimensions   = 2,
    offset       = [-0.1, -0.1],
    size         = [0.2, 0.2],
    min_end_time = 0.02,
    max_end_time = 0.02,
    periodic_BC  = [True, True, True],
)

## Computing the decomposition offline

The offline load balancing works on a Python representation of the spacetree.
Our mesh is regular: With patches of $9 \times 9$ volumes of size 0.001, each
cell has to be smaller than 0.009. The spacetree hence is refined three times,
i.e. the finest level holds $27 \times 27 = 729$ cells. We do not have to read
this mesh from a file, as we can synthesise it:

In [ ]:
import toolbox.loadbalancing.api.input.RegularGridGenerator as RegularGridGenerator
import toolbox.loadbalancing.api.strategies.SplitTrees as SplitTrees
import toolbox.loadbalancing.api.strategies.Decomposition as Decomposition
import toolbox.loadbalancing.api.output.Hardcoded as Hardcoded

tree = RegularGridGenerator.RegularGridGenerator(max_level=3, dimensions=2)
print(tree)

Next, we split the tree. We want one subtree per thread, i.e. four subtrees.
The equidistant decomposition gives each subtree the same number of cells, and
split_tree() assigns the cells to the subtrees:

In [ ]:
number_of_subtrees = 4

SplitTrees.split_tree(
    tree,
    decomposition = Decomposition.equidistant_splitting(peano_tree=tree, number_of_subtrees=number_of_subtrees),
)

Finally, we map the subtrees onto ranks and translate the outcome into the
arguments of the hardcoded load balancing. As we run on one rank only, all
subtrees go to rank 0. Tree 0 is the tree that all other trees are split off
from, so we only need destination ranks for the other three trees:

In [ ]:
destination_ranks = Hardcoded.assign_trees_to_ranks_round_Robin(
    number_of_subtrees = tree.number_of_subtrees,
    max_ranks          = 1,
)

hardcoded_configuration, hardcoded_mode = Hardcoded.create_constructor_initialisation_string(
    tree, destination_ranks, verbose=True
)

toolbox::loadbalancing::strategies::Hardcoded takes four lists plus the split
mode. Entry $i$ of the lists describes the $i$th split:

- **grid_sweeps** holds the mesh sweep in which the split is triggered. The offline
  load balancing derives it from the level on which the new subtree starts, i.e.
  it uses $2 \cdot (\ell + 1)$ with the level $\ell$ counted from zero. All three
  splits hence happen in sweep 6, once the mesh has been constructed.
- **splitting_trees** holds the tree which splits. Here, all subtrees are forked
  off tree 0.
- **number_of_cells** holds the number of cells that the split hands over to the
  new tree, i.e. 182 cells each. Tree 0 keeps the remaining cells.
- **destination ranks** holds the rank which receives the new tree.

The mode AggressiveTopDown states that all subtrees start on the same level.
If you use more than one rank, pass the number of ranks as max_ranks, and map
the trees onto the ranks such that each rank gets the same number of trees.

The constructor string is all that we hand over to the project:

In [ ]:
print(hardcoded_configuration)

project.set_load_balancing(
    "toolbox::loadbalancing::strategies::Hardcoded",
    hardcoded_configuration,
)

If you prefer to write C++ yourself, e.g. within your own main, the call
above corresponds to

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#include "toolbox/loadbalancing/strategies/Hardcoded.h"

auto* loadBalancing = new toolbox::loadbalancing::strategies::Hardcoded(
  {6, 6, 6},          // grid sweeps
  {0, 0, 0},          // splitting trees
  {182, 182, 182},    // number of cells
  {0, 0, 0},          // destination ranks
  peano4::SplitInstruction::Mode::AggressiveTopDown
);
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

You can also pass one mode per split instead of one mode for all splits.

## Building and running the code

The remaining steps are the same as in @ref tutorials_exahype2_30_domain-decomposition:

In [ ]:
project.set_Peano4_installation(os.path.abspath("../../"), peano4.api.output.CompileMode.RELEASE)

peano4_project = project.generate_Peano4_project()
peano4_project.generate()

In [ ]:
%%writefile 35_offline-load-balancing/EulerFV.cpp
#include "EulerFV.h"


void tutorials::exahype2::offlineloadbalancing::EulerFV::flux(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ F
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));

  switch (normal) {
    case 0:
      F[0] = Q[1];
      F[1] = irho * Q[1] * Q[1] + p;
      F[2] = irho * Q[1] * Q[2];
      F[3] = irho * Q[1] * (Q[3] + p);
      break;
    case 1:
      F[0] = Q[2];
      F[1] = irho * Q[2] * Q[1];
      F[2] = irho * Q[2] * Q[2] + p;
      F[3] = irho * Q[2] * (Q[3] + p);
      break;
  }
}


void tutorials::exahype2::offlineloadbalancing::EulerFV::maxEigenvalue(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ lambdaMax
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));
  const double c = std::sqrt(gamma * p * irho);

  // velocity, i.e. momentum divided by density
  const double u = Q[1 + normal] * irho;

  lambdaMax[0] = std::max(std::abs(u + c), std::abs(u - c));
}


void tutorials::exahype2::offlineloadbalancing::EulerFV::initPointWisely(
  const tarch::la::Vector<Dimensions, double>& x, double* __restrict__ Q
) {
  // One period of a sine density profile over the periodic domain
  // [-0.1,0.1], constant velocity u and constant pressure p
  constexpr double gamma = 1.4;
  constexpr double p     = 1.0;
  constexpr double u     = 1.0;
  const double     rho   = 1.0 + 0.2 * std::sin(2.0 * M_PI * (x[0] + 0.1) / 0.2);
  Q[0] = rho;
  Q[1] = rho * u;
  Q[2] = 0.0;
  Q[3] = p / (gamma - 1) + 0.5 * rho * u * u;
}

In [ ]:
!cd 35_offline-load-balancing && make -j > make.log 2>&1 && OMP_NUM_THREADS=4 ./euler > run.log 2>&1 && echo done

The log shows how the hardcoded load balancing hands out its splits, and the
final distribution of the cells among the trees:

In [ ]:
!grep -E "Hardcoded|dumpStatistics" 35_offline-load-balancing/run.log | tail -3

## Visualising the subdomains

We convert the output and open it in ParaView as described in
@ref tutorials_exahype2_30_domain-decomposition:

In [ ]:
!cd 35_offline-load-balancing && PYTHONPATH=../../../src python3 ../../../src/tarch/plotter/api/render.py solution-EulerFV.peano-patch-file > render.log 2>&1 && echo done

![The hardcoded subdomains of the Euler run in ParaView](illustrations/35_offline-load-balancing-subdomains.png)

The decomposition resembles the one from
@ref tutorials_exahype2_30_domain-decomposition, as both cut the same mesh
into four chunks along the space-filling curve. Again, tree 0 keeps a frame
along the whole periodic boundary.

The log however shows that the trees do not hold exactly 182 cells each, but
somewhere between 177 and 186 cells. The offline load balancing plans with
a tree in which each cell can be handed over to any other tree. Peano however
has to respect further constraints when it carries out a split. Notably, it
keeps all cells along periodic boundaries on tree 0. The splits therefore only
approximate the offline plan. It is hence always worth checking the actual
distribution in the log.

## Meshes that are not regular

For adaptive meshes, we cannot synthesise the spacetree. Instead, we run the
code once without load balancing, stop it once the mesh has been constructed,
and feed the resulting patch files into the offline load balancing. The script
OfflineDecomposition.py reads such patch files, splits the tree, and writes the
outcome into a file:

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
python3 ../../src/toolbox/loadbalancing/api/OfflineDecomposition.py -f my-mesh.peano-patch-file -d 2 -s 4
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Here, -d is the number of dimensions and -s the number of subtrees. Call the
script with --help to see all options.
The @ref tutorials_exahype2_hpc_offline-load-balancing "offline load balancing tutorial for the single black hole benchmark"
discusses this workflow and further decomposition variants in detail, and
@ref page_toolbox_loadbalancing_api "the load balancing API" documents the
underlying routines.